In [ ]:
import os
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
from pathlib import Path
import numpy as np


In [ ]:
BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR


In [ ]:
# --- Load grid geometry + Cell x Year wide table (if not already in memory) ---
grid_path = os.path.join(PROJECT_ROOT, "data", "raw", "ihr_grid_2_5km_wgs84.gpkg")
wide_path = os.path.join(PROJECT_ROOT, "data", "raw", "ihr_cell_year_wide.csv")

grid = gpd.read_file(grid_path)          # Cell_ID, cell_size_km, geometry
wide = pd.read_csv(wide_path)            # Cell_ID, 2001, 2002, ..., 2025

grid_data = grid.merge(wide, on="Cell_ID", how="left")


In [ ]:
# --- Set the year you want to see ---
YEAR_TO_PLOT = 2024  # <-- change this manually

fig, ax = plt.subplots(figsize=(10, 10))
grid_data.plot(
    column=str(YEAR_TO_PLOT),
    cmap="YlOrRd",
    linewidth=0.05,
    edgecolor="grey",
    legend=True,
    legend_kwds={"label": "Forest Loss (ha)", "shrink": 0.6},
    ax=ax,
    missing_kwds={"color": "lightgrey"},
)
ax.set_title(f"IHR Forest Loss by Cell - {YEAR_TO_PLOT}")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
'''out_dir = os.path.join(PROJECT_ROOT, "data", "processed", "forest_loss_cells_absolute")
os.makedirs(out_dir, exist_ok=True)

# Use a common color scale across all years so maps are visually comparable
year_cols = [str(y) for y in range(2001, 2026)]
vmin, vmax = 0, grid_data[year_cols].quantile(0.99).max()  # cap at 99th pct to avoid one outlier washing out the scale

for year in range(2001, 2026):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid_data.plot(
        column=str(year),
        cmap="YlOrRd",
        vmin=vmin,
        vmax=vmax,
        linewidth=0.05,
        edgecolor="grey",
        legend=True,
        legend_kwds={"label": "Forest Loss (ha)", "shrink": 0.6},
        ax=ax,
        missing_kwds={"color": "lightgrey"},
    )
    ax.set_title(f"IHR Forest Loss by Cell - {year}")
    ax.set_axis_off()

    fig_path = os.path.join(out_dir, f"forest_loss_{year}.png")
    fig.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.close(fig)  # important: prevents 25 open figures piling up in memory

    print(f"Saved {fig_path}")

print(f"\nDone - {len(range(2001, 2026))} maps written to {out_dir}")'''

## Relative Forest Loss export

In [ ]:
# --- Load 2000 baseline forest area ---
baseline_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw",
    "forest_loss_annual_yearly",
    "ihr_baseline_2000_combined.csv"
)
baseline = pd.read_csv(baseline_path)
# Keep only the columns we need
baseline = baseline[["Cell_ID", "Baseline_Forest_ha"]].copy()

# Make sure Cell_ID is unique
baseline = baseline.drop_duplicates("Cell_ID")

# --- Merge baseline forest area into grid_data ---
grid_data = grid_data.merge(
    baseline,
    on="Cell_ID",
    how="left"
)

print(f"Grid cells: {len(grid_data):,}")
print(f"Cells with baseline data: {grid_data['Baseline_Forest_ha'].notna().sum():,}")
print(f"Cells with zero baseline forest: {(grid_data['Baseline_Forest_ha'] == 0).sum():,}")

In [ ]:
# Annual forest-loss columns
year_cols = [str(y) for y in range(2001, 2026)]

# Create relative forest-loss (%) for every year
for year in year_cols:

    grid_data[f"{year}_relative"] = np.where(
        grid_data["Baseline_Forest_ha"] > 0,
        (grid_data[year] / grid_data["Baseline_Forest_ha"]) * 100,
        np.nan
    )

print("Relative forest-loss columns created:")
print([f"{y}_relative" for y in range(2001, 2026)])

In [ ]:
rel_col = "2005_relative"

print(
    grid_data[rel_col]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995
        ]
    )
)

In [ ]:
print("Maximum relative loss:", grid_data[rel_col].max())

print("\nTop 20 cells:")
print(
    grid_data[
        ["Cell_ID", "Baseline_Forest_ha", "2005", rel_col]
    ]
    .sort_values(rel_col, ascending=False)
    .head(20)
)

In [ ]:
# --- Set the year you want to see ---
YEAR_TO_PLOT = 2016 

rel_col = f"{YEAR_TO_PLOT}_relative"

# Use the 99th percentile as the upper colour limit
vmin = 0
vmax = grid_data[rel_col].quantile(0.99)

print(f"Colour scale: 0 to {vmax:.2f}%")

fig, ax = plt.subplots(figsize=(10, 10))

grid_data.plot(
    column=rel_col,
    cmap="YlOrRd",
    vmin=vmin,
    vmax=vmax,
    linewidth=0.05,
    edgecolor="grey",
    legend=True,
    legend_kwds={
        "label": "Relative Forest Loss (%)",
        "shrink": 0.6
    },
    ax=ax,
    missing_kwds={"color": "lightgrey"},
)

ax.set_title(
    f"IHR Relative Forest Loss by Cell - {YEAR_TO_PLOT}"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# --- Export relative forest-loss maps for 2001–2025 ---

out_dir = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "forest_loss_cells_relative"
)

os.makedirs(out_dir, exist_ok=True)

# Relative-loss columns
relative_cols = [
    f"{year}_relative"
    for year in range(2001, 2026)
]

# Common colour scale across all years
# 99th percentile prevents a few extreme cells
# from dominating the visualisation.
vmin = 0
vmax = grid_data[relative_cols].quantile(0.99).max()

print(f"Common colour scale: 0 to {vmax:.2f}%")

for year in range(2001, 2026):

    rel_col = f"{year}_relative"

    fig, ax = plt.subplots(figsize=(10, 10))

    grid_data.plot(
        column=rel_col,
        cmap="YlOrRd",
        vmin=vmin,
        vmax=vmax,
        linewidth=0.05,
        edgecolor="grey",
        legend=True,
        legend_kwds={
            "label": "Relative Forest Loss (%)",
            "shrink": 0.6
        },
        ax=ax,
        missing_kwds={"color": "lightgrey"},
    )

    ax.set_title(
        f"IHR Relative Forest Loss by Cell - {year}"
    )
    ax.set_axis_off()

    fig_path = os.path.join(
        out_dir,
        f"relative_forest_loss_{year}.png"
    )

    fig.savefig(
        fig_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"Saved: {fig_path}")

print(
    f"\nDone — {len(range(2001, 2026))} maps exported to:\n{out_dir}"
)